In [1]:
import json
import sys
from pathlib import Path

# Resolve sys.path to the t12_pipeline src/ so the package imports work
# regardless of where the notebook is launched from.
_HERE = Path.cwd().resolve()
_SRC = next(
    (p for p in [_HERE, *_HERE.parents] if (p / 't12_pipeline' / 'config' / 'paths.py').is_file()),
    None,
)
if _SRC and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

from t12_pipeline.config.paths import (
    DB_RULEBOOKS_DIR,
    MAPPED_T12_DOCS_DIR,
    OUTPUT_DOCS_BASE_DIR,
)
from t12_pipeline.config.rulebook_utils import get_rulebook_categories

# The active rulebook is the newest mapping_rulebook_*.json in db/rulebooks/.
# Same logic both the pipeline and the UI bridge use to pick the rulebook.
_candidates = sorted(DB_RULEBOOKS_DIR.glob('mapping_rulebook_*.json'))
if not _candidates:
    raise FileNotFoundError(
        f"No rulebook in {DB_RULEBOOKS_DIR}. "
        "Run `python -m t12_pipeline.build_rulebook.run` first."
    )
LATEST_RULEBOOK_PATH = _candidates[-1]

print(f'Latest rulebook: {LATEST_RULEBOOK_PATH.name}')
print(f'Mapped docs dir: {MAPPED_T12_DOCS_DIR}')

Latest rulebook: mapping_rulebook_2026_06_18_0321.json
Mapped docs dir: /Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/app/t12_pipeline/data/mapped_T12_docs


## Section 1 — Category Coverage Check

Checks which expected canonical categories are present or missing in the active v2 rulebook.
Update `expected_keys` below if the chart of accounts changes.

In [2]:
# Glob both .xlsm and .xlsx — matches what extract_reviewed_row_history.py uses.
files = sorted(
    list(MAPPED_T12_DOCS_DIR.glob('*.xlsm'))
    + list(MAPPED_T12_DOCS_DIR.glob('*.xlsx'))
)
files = [f for f in files if not f.name.startswith('~$')]

# Show the split between .xlsm and .xlsx for visibility
xlsm_files = [f for f in files if f.suffix.lower() == '.xlsm']
xlsx_files = [f for f in files if f.suffix.lower() == '.xlsx']

print(f'Training files ({len(files)} total: {len(xlsm_files)} .xlsm + {len(xlsx_files)} .xlsx):')
for f in files:
    print(f'  {f.name}')

Training files (42 total: 24 .xlsm + 18 .xlsx):
  02242026_Grande Reserve at Geist_V3- VM..xlsx
  02242026_Grande Reserve at Geist_V3- VM.xlsm
  02252026_Willows of Coventry_V4 (1).xlsm
  03092026_The Flats at Arrowood_V4 (FULL BUDGET).xlsm
  03092026_The Tatum_V4 (assumption).xlsm
  030926_Tides At Eastchase_v3.xlsm
  031026_Mylo Apartment Homes_V2 - VM.xlsm
  03112026_Forma at the Park_V1.xlsm
  03112026_The Rivi_V6 - AW Capex Notes .xlsm
  03122025_District at Hurstborne_V2.xlsm
  03182026_The Park at LeBlanc_V3- VM.xlsm
  03192026_The Jasmine Apartments_V1.xlsm
  03192026_The Jasmine Apartments_V2 - VM.xlsm
  03192026_The Park at LeBlanc_V4.1 - AW CapEx.xlsm
  03242026_Prescott Park Apartments_V1.5.xlsm
  03272026_Devon on Northgate_V1.xlsm
  04032026_Villa Del Mar_V1.5.xlsm
  080425_Jamestown at St. Matthews_v4.xlsm
  080825_Kenton Reserve_v4.xlsm
  100325_Park at Eagle Creek_v3.xlsx
  11182025_Exchange 7272_V1.xlsm
  11182025_The Hangar_V1.xlsm
  12042025_The Hathaway at Willow B

In [3]:
# Full canonical chart of accounts provided by the client.
# Every account type in the raw T12 data should get mapped to one of these.
# (Matching is case-insensitive — preserve the client's exact spelling here.)
expected_keys = [
    'SUBTOTAL / IGNORE',
    'Gross Potential Rent',
    '(Loss) Gain to Lease',
    'Rental Concessions',
    'Vacancy Loss',
    'Non Revenue Units',
    'Bad debt',
    'Utility Rebill - Electric',
    'Utility Rebill - Gas',
    'Utility Rebill - Water/Sewer',
    'Utility Rebill - Trash Removal',
    'Utility Rebill - Cable, Internet, Other',
    'Utility Rebill - Other',
    'Other Income Ops',
    'Parking Income',
    'Laundry and Vending',
    'Commercial Rents - Net',
    'Repairs and Maintenance',
    'Turnover Expenses',
    'Contracted Services',
    'Professional Fees',
    'Payroll Salaries/Wages',
    'Payroll Burden',
    'Utilities Electric',
    'Utilities Gas',
    'Utilities Water/Sewer',
    'Utilities Trash Removal',
    'Utilities Cable, Internet, Other',
    'Utilities Other',
    'Administrative',
    'Marketing and Retention',
    'Management Fee',
    'Miscellaneous Operating Expenses',
    'Real Estate Taxes',
    'Insurance',
    'Ground Lease',
    'Capex Ongoing',
    'Capex Rehab / Replace',
    'Capex Major',
    'Capex Reserve',
    'Capex Special',
    'Capex Commercial',
    'Debt Service Interest',
    'Debt Origination',
    'Other',
    'Other Income: Value Add',
]

with open(LATEST_RULEBOOK_PATH, 'r') as f:
    rulebook = json.load(f)

# Current rulebook shape: { certain, contextual, no_conflict, unresolved }
# get_rulebook_categories pulls the union of canonical categories actually used.
categories = sorted(get_rulebook_categories(rulebook))
categories_lower = {c.strip().lower() for c in categories}

present = [k for k in expected_keys if k.strip().lower() in categories_lower]
missing = [k for k in expected_keys if k.strip().lower() not in categories_lower]
extras  = [c for c in categories if c.strip().lower() not in {k.strip().lower() for k in expected_keys}]

print(f'Rulebook source: {LATEST_RULEBOOK_PATH.name}')
print(f'  Buckets:           certain={len(rulebook.get("certain", {}))}, '
      f'contextual={len(rulebook.get("contextual", {}))}, '
      f'no_conflict={len(rulebook.get("no_conflict", {}))}, '
      f'unresolved={len(rulebook.get("unresolved", {}))}')
print(f'  Unique categories: {len(categories)}')
print()
print(f'Expected (client list): {len(expected_keys)}')
print(f'  Present: {len(present)}')
print(f'  Missing: {len(missing)}')

if missing:
    print('\nMissing from rulebook (client expects but no row maps to them yet):')
    for m in missing:
        print(f'  - {m}')
else:
    print('\n✓ All expected client categories are present in the rulebook.')

if extras:
    print(f'\nExtra categories in rulebook (not in client list — possible spelling/capitalization drift): {len(extras)}')
    for e in extras:
        print(f'  - {e}')

print('\nAll categories in rulebook:')
for k in categories:
    print(f'  {k}')

Rulebook source: mapping_rulebook_2026_06_18_0321.json
  Buckets:           certain=72, contextual=10, no_conflict=2638, unresolved=79
  Unique categories: 38

Expected (client list): 46
  Present: 38
  Missing: 8

Missing from rulebook (client expects but no row maps to them yet):
  - SUBTOTAL / IGNORE
  - Miscellaneous Operating Expenses
  - Ground Lease
  - Capex Reserve
  - Capex Special
  - Capex Commercial
  - Debt Service Interest
  - Debt Origination

All categories in rulebook:
  (Loss) Gain To Lease
  Administrative
  Bad Debt
  Capex Major
  Capex Ongoing
  Capex Rehab / Replace
  Commercial Rents - Net
  Contracted Services
  Gross Potential Rent
  Insurance
  Laundry And Vending
  Management Fee
  Marketing And Retention
  Non Revenue Units
  Other
  Other Income Ops
  Other Income: Value Add
  Parking Income
  Payroll Burden
  Payroll Salaries/Wages
  Professional Fees
  Real Estate Taxes
  Rental Concessions
  Repairs And Maintenance
  Turnover Expenses
  Utilities Cable

## Section 2 — AI Mapping Log Reader

Reads and prints the full LLM prompt and raw response from an AI mapping log file.
Useful for inspecting what context the model received and how it responded.

Change `log_file` below to inspect a different property.

In [4]:
# The setup cell at the top already imported what we need
# (OUTPUT_DOCS_BASE_DIR). Section 2 reuses that scope.

In [5]:
# Walk every past pipeline run and list AI-mapping log files.
# AI_MAPPING_LOGS_DIR in paths.py points to the CURRENT timestamp only — but we
# want to inspect any past run, so glob through output_docs/<ts>/2_mapping/ai_mapping_logs.
log_dirs = sorted(OUTPUT_DOCS_BASE_DIR.glob('*/2_mapping/ai_mapping_logs'))

if not log_dirs:
    print(f'No AI mapping logs found yet under {OUTPUT_DOCS_BASE_DIR}/*/2_mapping/ai_mapping_logs/')
    print('Run the pipeline first: python -m t12_pipeline.pipeline.run_pipeline --stages 0,1,2')
else:
    print(f'Found {len(log_dirs)} run(s) with AI mapping logs:\n')
    for d in log_dirs:
        run_id = d.parent.parent.name
        log_files = sorted(d.glob('*.json'))
        print(f'  {run_id}  ({len(log_files)} log files)')
        for f in log_files:
            print(f'    {f.name}')

Found 6 run(s) with AI mapping logs:

  2026_05_18_0907  (7 log files)
    Brownsburg Crossing T12 05-25_logs.json
    Carmel Woods T12 09-25_logs.json
    Echo Park at Perry Crossing T12 12-24_logs.json
    Pebble Brook Village T12 03-25_logs.json
    Pebble Brook Village T12 09-25_logs.json
    Pointe Inverness T12 11-25_logs.json
    Willows of Coventry T12 11-25_logs.json
  2026_05_18_1024  (0 log files)
  2026_05_19_1118  (0 log files)
  2026_05_22_1045  (0 log files)
  2026_05_22_1052  (0 log files)
  2026_05_22_1053  (0 log files)


In [6]:
# Default: pick the most recent run's first log file.
# Skip any run dir whose ai_mapping_logs/ exists but is empty (early build_rulebook
# runs that didn't reach Stage 2 leave empty dirs around).
#
# To inspect a specific log, override `log_file` explicitly, e.g.:
#   log_file = OUTPUT_DOCS_BASE_DIR / '2026_06_18_0259' / '2_mapping' / 'ai_mapping_logs' / 'Brownsburg Crossing T12 05-25_logs.json'

if not log_dirs:
    raise FileNotFoundError('No AI mapping logs available — run the pipeline first.')

# Walk newest-first, take the first dir that actually has .json log files.
log_file = None
for d in reversed(log_dirs):
    candidates = sorted(d.glob('*.json'))
    if candidates:
        log_file = candidates[0]
        break

if log_file is None:
    raise FileNotFoundError(
        f'All {len(log_dirs)} ai_mapping_logs/ dirs are empty. '
        'Run Stage 2 (mapping) of the pipeline first: '
        'python -m t12_pipeline.pipeline.run_pipeline --stages 0,1,2'
    )

with open(log_file, 'r') as f:
    log = json.load(f)

print(f'Log file: {log_file.relative_to(OUTPUT_DOCS_BASE_DIR)}')
print(f'Entries:  {len(log)}')

Log file: 2026_05_18_0907/2_mapping/ai_mapping_logs/Brownsburg Crossing T12 05-25_logs.json
Entries:  3


In [7]:
# Print the first entry — change index to inspect other rows
for entry in log:
# entry = log[0]
    print('=== PROMPT ===')
    print(entry.get('llm_final_prompt'))
    
    print('\n=== RAW OUTPUT ===')
    print(json.dumps(entry.get('llm_raw_output'), indent=2))

=== PROMPT ===
You are an expert real estate financial accountant working inside the Shivam Excel AI Automation pipeline.

Your job: a T12 (Trailing-12-Month) property financial statement has a line item that the primary rulebook could not automatically map. Analyze it and choose the correct canonical category.

---
### Fields in item_to_map

- `description` — exact line item label from the T12 file
- `statement_side` — structural position detected from the raw file layout:
    "revenue"  → row sits in the income section (before the grand income total)
    "expense"  → row sits in the operating expense section (after income total, before NOI)
  Trust this field — it is derived from file structure, not text. A row labeled "Bad Debt Expense" that appears in the income section gets statement_side="revenue" because it is a contra-revenue adjustment there, not an operating expense.
- `section_context` — immediate section header directly above this row
- `section_path` — full header hierarch

## Section 3 — Rulebook diff (latest vs previous build)

Compares the newest `mapping_rulebook_*.json` in `db/rulebooks/` against the second-newest one. Useful right after a rebuild to see what grew:

- Total descriptions before / after
- Per-bucket counts (certain / contextual / no_conflict / unresolved)
- **New** descriptions added in this build
- **New** canonical categories used
- Descriptions that **moved buckets** (e.g. unresolved → contextual after a client review update)
- Descriptions that **dropped** (rare — usually 0)

Filenames sort chronologically (timestamp format), so the last two by name = latest + previous.

In [8]:
BUCKETS = ("certain", "contextual", "no_conflict", "unresolved")


def _flatten(rulebook: dict) -> dict[str, dict]:
    """{description: {'bucket': str, 'category': str|None}} for every entry."""
    flat = {}
    for bucket in BUCKETS:
        for key, entry in (rulebook.get(bucket) or {}).items():
            # Pull whichever category-like field exists for this bucket.
            cat = None
            if isinstance(entry, dict):
                cat = (
                    entry.get("category")
                    or entry.get("mapping")
                    or entry.get("final_mapping")
                    or entry.get("auto_mapping")
                )
            flat[key] = {"bucket": bucket, "category": cat}
    return flat


def _bucket_counts(rulebook: dict) -> dict[str, int]:
    return {b: len(rulebook.get(b) or {}) for b in BUCKETS}


def _categories(rulebook: dict) -> set[str]:
    """All canonical categories used anywhere in the rulebook."""
    return set(get_rulebook_categories(rulebook))


# Pick the two most recent rulebooks by filename (timestamp-sorted)
rulebooks_sorted = sorted(DB_RULEBOOKS_DIR.glob("mapping_rulebook_*.json"))
if len(rulebooks_sorted) < 2:
    raise ValueError(
        f"Need at least 2 rulebooks in {DB_RULEBOOKS_DIR} to diff; found {len(rulebooks_sorted)}."
    )

prev_path = rulebooks_sorted[-2]
latest_path = rulebooks_sorted[-1]

with open(prev_path) as f:
    prev_rb = json.load(f)
with open(latest_path) as f:
    latest_rb = json.load(f)

print(f"Previous: {prev_path.name}")
print(f"Latest:   {latest_path.name}")

Previous: mapping_rulebook_2026_06_18_0316.json
Latest:   mapping_rulebook_2026_06_18_0321.json


In [9]:
# --- Bucket counts before / after ---------------------------------------------
prev_counts = _bucket_counts(prev_rb)
latest_counts = _bucket_counts(latest_rb)

print(f"{'Bucket':<14} {'Previous':>10} {'Latest':>10} {'Delta':>8}")
print("-" * 46)
prev_total = sum(prev_counts.values())
latest_total = sum(latest_counts.values())
for b in BUCKETS:
    d = latest_counts[b] - prev_counts[b]
    arrow = "+" if d > 0 else ""
    print(f"{b:<14} {prev_counts[b]:>10} {latest_counts[b]:>10} {arrow}{d:>7}")
print("-" * 46)
print(f"{'TOTAL':<14} {prev_total:>10} {latest_total:>10} "
      f"{'+' if latest_total - prev_total > 0 else ''}{latest_total - prev_total:>7}")

Bucket           Previous     Latest    Delta
----------------------------------------------
certain                72         72       0
contextual             10         10       0
no_conflict          2638       2638       0
unresolved             79         79       0
----------------------------------------------
TOTAL                2799       2799       0


In [10]:
# --- New categories used ------------------------------------------------------
prev_cats = _categories(prev_rb)
latest_cats = _categories(latest_rb)
new_cats = sorted(latest_cats - prev_cats)
dropped_cats = sorted(prev_cats - latest_cats)

print(f"Unique categories: previous={len(prev_cats)}  latest={len(latest_cats)}\n")

if new_cats:
    print(f"NEW canonical categories used ({len(new_cats)}):")
    for c in new_cats:
        print(f"  + {c}")
else:
    print("No new categories.")

if dropped_cats:
    print(f"\nDropped categories ({len(dropped_cats)}):")
    for c in dropped_cats:
        print(f"  - {c}")

Unique categories: previous=38  latest=38

No new categories.


In [11]:
# --- New descriptions, dropped, and bucket movements --------------------------
prev_flat = _flatten(prev_rb)
latest_flat = _flatten(latest_rb)

added_keys   = sorted(set(latest_flat) - set(prev_flat))
dropped_keys = sorted(set(prev_flat) - set(latest_flat))
moved = [
    (k, prev_flat[k]["bucket"], latest_flat[k]["bucket"])
    for k in (set(prev_flat) & set(latest_flat))
    if prev_flat[k]["bucket"] != latest_flat[k]["bucket"]
]
moved.sort()

print(f"Added descriptions:   {len(added_keys)}")
print(f"Dropped descriptions: {len(dropped_keys)}")
print(f"Moved buckets:        {len(moved)}\n")

if added_keys:
    # Group by bucket for readability
    from collections import defaultdict
    by_bucket = defaultdict(list)
    for k in added_keys:
        by_bucket[latest_flat[k]["bucket"]].append(k)
    print("--- NEW descriptions in this build (grouped by bucket) ---")
    for b in BUCKETS:
        keys = by_bucket.get(b, [])
        if not keys:
            continue
        print(f"\n  [{b}] {len(keys)} new")
        for k in keys[:20]:
            cat = latest_flat[k]["category"] or "-"
            print(f"    + {k}   →   {cat}")
        if len(keys) > 20:
            print(f"    ... and {len(keys) - 20} more")

if moved:
    print("\n--- Descriptions that changed bucket ---")
    for k, before, after in moved[:30]:
        print(f"  {k}: {before} → {after}")
    if len(moved) > 30:
        print(f"  ... and {len(moved) - 30} more")

if dropped_keys:
    print("\n--- Descriptions that DISAPPEARED (usually a bug) ---")
    for k in dropped_keys[:20]:
        print(f"  - {k}  (was {prev_flat[k]['bucket']})")
    if len(dropped_keys) > 20:
        print(f"  ... and {len(dropped_keys) - 20} more")

Added descriptions:   0
Dropped descriptions: 0
Moved buckets:        0



## Section 4 — Parser Format Audit

Survey a folder of `.xlsx` files and bucket each one. Useful when the
client sends a new batch — see at a glance which files our pipeline
can parse today, which need new format support, and which aren't T12
inputs at all.

Each file lands in exactly one of these buckets:

| Bucket | Meaning | Will the pipeline parse it? |
|---|---|---|
| `known:code_in_a_desc_in_b` | Recognized layout | yes |
| `known:embedded_code_in_a`  | Recognized layout | yes |
| `known:no_code`             | Recognized layout | yes |
| `unknown_t12_format`        | Looks like a T12 but the layout isn't one of the three we support | no — needs new format / AI |
| `rent_roll`                 | Filename or sheet name says rent roll | no — different document type |
| `model_t12`                 | Filename ends with "Model T12.xlsx" — canonical target, not input | no — not meant as an input |
| `errored`                   | Couldn't open the workbook | no |

Bucketing order (first match wins):
1. Filename ends with `Model T12.xlsx` → `model_t12`
2. Filename matches rent-roll pattern (`RR`, `Rent Roll`) → `rent_roll`
3. Any sheet name matches rent-roll pattern → `rent_roll`
4. `detect_format()` returns one of the three known formats → `known:<format>`
5. Otherwise → `unknown_t12_format`

Read-only — doesn't write any pipeline output.

Tweak `SCAN_DIR` below to point at the folder you want to audit.

In [12]:
import io
import contextlib
import re
import openpyxl

from t12_pipeline.pipeline.detection.detect_blocks import detect_format

# ── Tweak this ────────────────────────────────────────────────────────────────
SCAN_DIR = Path(
    "/Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/"
    "master_docs/client_sent_docs/6th_june_new_t12_docs/T12"
)
RECURSIVE = True   # set False to scan only the top level of SCAN_DIR

# ── Pre-classifier patterns ──────────────────────────────────────────────────
# Run BEFORE detect_format() so we separate "not a T12 input at all" from
# "is a T12 but uses a layout we don't recognize yet".
RENT_ROLL_RE = re.compile(r"(?:\bRR\b|_RR_|Rent[\s_]?Roll)", re.IGNORECASE)
MODEL_T12_RE = re.compile(r"Model[\s_]?T12\.xlsx$", re.IGNORECASE)

KNOWN_FORMATS = ("code_in_a_desc_in_b", "embedded_code_in_a", "no_code")


def bucket_for(fp: Path) -> tuple[str, str | None]:
    """Return (bucket, error_msg). bucket is one of:
       known:<format> | unknown_t12_format | rent_roll | model_t12 | errored"""
    name = fp.name

    if MODEL_T12_RE.search(name):
        return ("model_t12", None)
    if RENT_ROLL_RE.search(name):
        return ("rent_roll", None)

    # Sheet-name check (catches rent rolls with non-obvious filenames).
    try:
        with contextlib.redirect_stderr(io.StringIO()):
            wb = openpyxl.load_workbook(fp, data_only=True, read_only=True)
            sheet_names = list(wb.sheetnames)
            wb.close()
    except Exception as e:
        return ("errored", f"{type(e).__name__}: {e}")

    if any(RENT_ROLL_RE.search(s) for s in sheet_names):
        return ("rent_roll", None)

    # Otherwise — run the format detector on the active sheet.
    try:
        with contextlib.redirect_stderr(io.StringIO()):
            wb = openpyxl.load_workbook(fp, data_only=True)
            fmt = detect_format(wb.active)
            wb.close()
        if fmt in KNOWN_FORMATS:
            return (f"known:{fmt}", None)
        return ("unknown_t12_format", None)
    except Exception as e:
        return ("errored", f"{type(e).__name__}: {e}")


# ── Scan ──────────────────────────────────────────────────────────────────────
glob_method = SCAN_DIR.rglob if RECURSIVE else SCAN_DIR.glob
files = sorted(p for p in glob_method("*.xlsx") if not p.name.startswith("~$"))

print(f"Scanning: {SCAN_DIR}")
print(f"Found {len(files)} .xlsx files\n")

# Display order: working → broken → not-a-T12 → errored
DISPLAY_ORDER = [
    ("known:code_in_a_desc_in_b", "yes — known format"),
    ("known:embedded_code_in_a",  "yes — known format"),
    ("known:no_code",             "yes — known format"),
    ("unknown_t12_format",        "no  — looks like a T12 but unknown layout"),
    ("rent_roll",                 "no  — rent roll (not a T12)"),
    ("model_t12",                 "no  — Model T12 (canonical target, not an input)"),
    ("errored",                   "no  — file couldn't be opened"),
]

results: dict[str, list] = {b: [] for b, _ in DISPLAY_ORDER}

for fp in files:
    bucket, err = bucket_for(fp)
    rel = fp.relative_to(SCAN_DIR)
    if bucket == "errored":
        results["errored"].append((rel, err))
    else:
        results[bucket].append(rel)

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"{'Bucket':<32} {'Count':>6}  Note")
print("-" * 90)
for bucket, note in DISPLAY_ORDER:
    print(f"{bucket:<32} {len(results[bucket]):>6}  {note}")

# Coverage line: of the files that ARE T12 inputs (everything except rent rolls,
# model files, errored), how many does the pipeline parse today?
known_n   = sum(len(results[b]) for b, _ in DISPLAY_ORDER if b.startswith("known:"))
unknown_n = len(results["unknown_t12_format"])
t12_total = known_n + unknown_n
coverage  = (known_n / t12_total * 100) if t12_total else 0.0
print()
print(f"T12 input coverage: {known_n}/{t12_total} parse cleanly ({coverage:.0f}%); "
      f"{unknown_n} need new-format support.")

# ── Per-bucket file listing ───────────────────────────────────────────────────
for bucket, _ in DISPLAY_ORDER:
    items = results[bucket]
    if not items:
        continue
    print(f"\n── {bucket} ({len(items)}) ──")
    for item in items:
        if bucket == "errored":
            name, msg = item
            print(f"  {name}   [{msg[:80]}]")
        else:
            print(f"  {item}")

Scanning: /Users/manikantamb/_MyFiles/Work_/Freelancing/Shivam_Excel_AI_Automation/master_docs/client_sent_docs/6th_june_new_t12_docs/T12
Found 62 .xlsx files

Bucket                            Count  Note
------------------------------------------------------------------------------------------
known:code_in_a_desc_in_b             4  yes — known format
known:embedded_code_in_a              9  yes — known format
known:no_code                         4  yes — known format
unknown_t12_format                    8  no  — looks like a T12 but unknown layout
rent_roll                            21  no  — rent roll (not a T12)
model_t12                            16  no  — Model T12 (canonical target, not an input)
errored                               0  no  — file couldn't be opened

T12 input coverage: 17/25 parse cleanly (68%); 8 need new-format support.

── known:code_in_a_desc_in_b (4) ──
  New/Lincoln Townhomes Seller T12.xlsx
  Previously Provided/Carmel Woods T12 09-25.xlsx
  Previo